# My Orbit Wars Agent Lab

This notebook is a clean working copy for improving on `orbit-wars-step-by-step-agent-dev-ablation.ipynb`.

The guide's strongest final agent is kept as a baseline. My experimental agent adds:

- ROI target scoring that balances production, capture cost, distance, ownership, and enemy pressure.
- Source reserves so high-production planets are not stripped bare after every launch.
- Lightweight defensive reinforcement when an owned planet appears threatened.
- The same sun-avoidance and in-flight target reservation ideas from the guide.

The competition-facing function is `agent(obs)`.

In [1]:
import math
import random

try:
    import numpy as np
    import pandas as pd
    from tqdm import tqdm
    from kaggle_environments import make
except Exception:
    # Kaggle submission only needs the agent and standard library helpers.
    np = None
    pd = None
    tqdm = None
    make = None

random.seed(42)
if np is not None:
    np.random.seed(42)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

## Core Helpers

In [2]:
# Gets the speed of a fleet based on the number of ships
# Formula: speed scales from 1.0 (single ship) to max_speed (large fleets) using log scale
def fleet_speed(num_ships, max_speed=6.0):
    num_ships = max(1, int(num_ships))
    ratio = math.log(num_ships) / math.log(1000)
    ratio = max(0.0, ratio)
    return 1.0 + (max_speed - 1.0) * (ratio ** 1.5)

# Get the difference between 2 angles (in radians)
# Returns absolute angular distance, accounting for wraparound at 2π
def angle_diff(a, b):
    return abs(math.atan2(math.sin(a - b), math.cos(a - b)))

# Calculate Euclidean distance between 2 points on the map
# Planets are represented as [id, owner, x, y, ...]
def distance(a, b):
    return math.hypot(a[2] - b[2], a[3] - b[3])

 
# Split planets into my_planets (owned by current player) and targets (owned by others)
def split_planets(obs):
    player = obs["player"]
    planets = obs["planets"]
    my_planets = [p for p in planets if p[1] == player]
    targets = [p for p in planets if p[1] != player]
    return my_planets, targets


# Calculate the angle (in radians) from source planet to target planet.
# Used as a stationary fallback for fleet direction and navigation.
def get_angle(source, target):
    return math.atan2(target[3] - source[3], target[2] - source[2])


# Predict where a planet will be after a fractional number of turns.
# Regular planets rotate around the sun; static planets stay put; comets follow their path.
def planet_xy_at(planet, obs, turns_ahead=0.0):
    if turns_ahead <= 0:
        return planet[2], planet[3]

    pid = planet[0]
    for group in obs.get("comets", []):
        if pid in group.get("planet_ids", []):
            idx = group.get("planet_ids", []).index(pid)
            path = group.get("paths", [[]])[idx]
            path_index = group.get("path_index", -1)
            future_index = max(
                0,
                min(len(path) - 1, path_index + int(math.ceil(turns_ahead))),
            )
            if path:
                return path[future_index][0], path[future_index][1]

    initial_by_id = {p[0]: p for p in obs.get("initial_planets", [])}
    initial = initial_by_id.get(pid)
    angular_velocity = obs.get("angular_velocity")
    if initial is None or angular_velocity is None:
        return planet[2], planet[3]

    dx = initial[2] - 50.0
    dy = initial[3] - 50.0
    orbital_radius = math.hypot(dx, dy)
    if orbital_radius + planet[4] >= 50.0:
        return planet[2], planet[3]

    initial_angle = math.atan2(dy, dx)
    future_step = obs.get("step", 0) + turns_ahead
    future_angle = initial_angle + angular_velocity * future_step
    return (
        50.0 + orbital_radius * math.cos(future_angle),
        50.0 + orbital_radius * math.sin(future_angle),
    )


# Aim at the target's predicted intercept position instead of its current position.
# The travel time depends on the aim point, so iterate a few times to settle it.
def get_lead_angle(source, target, ships_to_send, obs, lead_scale=1.0, iterations=5):
    speed = fleet_speed(max(1, ships_to_send))
    turns = distance(source, target) / speed

    for _ in range(iterations):
        tx, ty = planet_xy_at(target, obs, turns * lead_scale)
        turns = math.hypot(tx - source[2], ty - source[3]) / speed

    tx, ty = planet_xy_at(target, obs, turns * lead_scale)
    return math.atan2(ty - source[3], tx - source[2])


# Find the closest target planet to a source planet by Euclidean distance
def get_nearest_target(source, targets):
    return min(targets, key=lambda t: distance(source, t))


# Check if a straight-line path from source at given angle passes through the sun
# Returns True if the path gets too close to the sun (within sun_radius + buffer)
def path_hits_sun(source, angle, sun_center=(50.0, 50.0), sun_radius=10.0, buffer=0.5):
    sx, sy = source[2], source[3]
    cx, cy = sun_center
    dx, dy = math.cos(angle), math.sin(angle)
    vx, vy = cx - sx, cy - sy

    # Project sun center onto the ray to find closest approach
    t = vx * dx + vy * dy
    if t <= 0:
        return False

    closest_x = sx + t * dx
    closest_y = sy + t * dy
    return math.hypot(closest_x - cx, closest_y - cy) <= sun_radius + buffer


## Fleet Target Inference

Orbit Wars fleets do not expose a direct target id, so the guide infers targets by comparing fleet direction to planet bearings. I kept that pattern and made it reusable for offense, radar, and defense.

In [3]:
# Infer which planet a fleet is targeting by comparing its heading to planet bearings
# Returns the best-matching planet if angle difference is within threshold, else None
# Used because Orbit Wars doesn't directly expose fleet target IDs
def infer_fleet_target(fleet, planets, angle_threshold=0.12, obs=None):
    fx, fy = fleet[2], fleet[3]
    fleet_angle = fleet[4]
    origin_id = fleet[5]

    best_planet = None
    best_diff = float("inf")

    for planet in planets:
        if planet[0] == origin_id:
            continue

        pseudo_source = [-1, fleet[1], fx, fy, 0, fleet[6], 0]
        if obs is None:
            target_angle = math.atan2(planet[3] - fy, planet[2] - fx)
        else:
            target_angle = get_lead_angle(pseudo_source, planet, fleet[6], obs)
        diff = angle_diff(target_angle, fleet_angle)

        if diff < best_diff:
            best_diff = diff
            best_planet = planet

    if best_planet is None or best_diff >= angle_threshold:
        return None

    return best_planet


# Get all planets that the player's own fleets are currently targeting
# Returns a set of planet IDs to avoid assigning new fleets to the same targets
# If use_capture_filter=True, only include targets that the fleet can actually capture
def get_reserved_targets(obs, angle_threshold=0.1, use_capture_filter=False):
    player = obs["player"]
    planets = obs["planets"]
    reserved = set()

    for fleet in obs["fleets"]:
        if fleet[1] != player:
            continue

        target = infer_fleet_target(fleet, planets, angle_threshold=angle_threshold, obs=obs)
        if target is None:
            continue

        if use_capture_filter and fleet[6] <= target[5]:
            continue

        reserved.add(target[0])

    return reserved


# Count incoming ship totals for each planet across all fleets
# owner: filter by fleet owner (-1 for any, or specific player ID)
# enemy_only: if True, only count enemy fleets
# Returns dict mapping planet_id -> total incoming ships
def incoming_ships_by_planet(obs, owner=None, enemy_only=False, angle_threshold=0.12):
    player = obs["player"]
    planets = obs["planets"]
    incoming = {p[0]: 0 for p in planets}

    for fleet in obs["fleets"]:
        if enemy_only and fleet[1] == player:
            continue
        if owner is not None and fleet[1] != owner:
            continue

        target = infer_fleet_target(fleet, planets, angle_threshold=angle_threshold, obs=obs)
        if target is not None:
            incoming[target[0]] = incoming.get(target[0], 0) + fleet[6]

    return incoming


# Estimate the defense strength of a target when the attack fleet arrives
# Accounts for the target's current ships, production during travel time, and planet growth
# Returns expected ship count at target when attack arrives
def estimate_target_defense(source, target, ships_to_send):
    arrival_turns = distance(source, target) / fleet_speed(max(1, ships_to_send))
    return math.ceil(target[5] + target[6] * arrival_turns)


## Guide Baseline

This is the late-stage guide agent: early no-reservation, enemy margin, estimated defense, enemy radar, value-based targeting, and sun avoidance.

In [4]:
# Score targets based on production value relative to distance or capture cost
# score_type options:
#   "prod_dist": production / distance (greedy by value per unit distance)
#   "prod_ship_dist": production / (ships * distance) (accounts for defender strength)
#   "prod_over_ships": production / ships (just value per defensive cost)
#   "nearest": -distance (priority by proximity)
# Returns the target with the highest score
def get_best_value_target(source, targets, score_type="prod_dist", eps=1e-6):
    def score(target):
        dist = distance(source, target)
        ships = target[5]
        prod = target[6]

        if score_type == "prod_dist":
            return prod / (dist + eps)
        if score_type == "prod_ship_dist":
            return prod / ((ships + 1) * (dist + eps))
        if score_type == "prod_over_ships":
            return prod / (ships + 1)
        if score_type == "nearest":
            return -dist

        raise ValueError(f"Unknown score_type: {score_type}")

    return max(targets, key=score)


# Stationary fleet-target inference used by the original guide baseline.
# V3 intentionally uses the lead-aware global helpers instead.
def infer_fleet_target_stationary(fleet, planets, angle_threshold=0.12):
    fx, fy = fleet[2], fleet[3]
    fleet_angle = fleet[4]
    origin_id = fleet[5]

    best_planet = None
    best_diff = float("inf")

    for planet in planets:
        if planet[0] == origin_id:
            continue

        target_angle = math.atan2(planet[3] - fy, planet[2] - fx)
        diff = angle_diff(target_angle, fleet_angle)

        if diff < best_diff:
            best_diff = diff
            best_planet = planet

    if best_planet is None or best_diff >= angle_threshold:
        return None

    return best_planet


def get_reserved_targets_stationary(obs, angle_threshold=0.1, use_capture_filter=False):
    player = obs["player"]
    planets = obs["planets"]
    reserved = set()

    for fleet in obs["fleets"]:
        if fleet[1] != player:
            continue

        target = infer_fleet_target_stationary(fleet, planets, angle_threshold=angle_threshold)
        if target is None:
            continue

        if use_capture_filter and fleet[6] <= target[5]:
            continue

        reserved.add(target[0])

    return reserved


def incoming_ships_by_planet_stationary(obs, owner=None, enemy_only=False, angle_threshold=0.12):
    player = obs["player"]
    planets = obs["planets"]
    incoming = {p[0]: 0 for p in planets}

    for fleet in obs["fleets"]:
        if enemy_only and fleet[1] == player:
            continue
        if owner is not None and fleet[1] != owner:
            continue

        target = infer_fleet_target_stationary(fleet, planets, angle_threshold=angle_threshold)
        if target is not None:
            incoming[target[0]] = incoming.get(target[0], 0) + fleet[6]

    return incoming


# Filter neutral targets to avoid those with heavy incoming enemy fire
# Always accepts owned planets (not neutral). For neutral planets, rejects if
# enemy incoming ships >= target's current ships + 1 (likely to be captured before arrival)
# This is "enemy radar" — detecting enemy intent to reduce wasted attacks
def filter_targets_by_enemy_radar(obs, targets, angle_threshold=0.15):
    enemy_incoming = incoming_ships_by_planet_stationary(obs, enemy_only=True, angle_threshold=angle_threshold)
    filtered = []

    for target in targets:
        if target[1] != -1:
            filtered.append(target)
            continue

        if enemy_incoming.get(target[0], 0) >= target[5] + 1:
            continue

        filtered.append(target)

    return filtered


# Factory function to create a configurable baseline agent
# Early-game no-reservation phase, then tracks reserved targets from own in-flight fleets
# Uses enemy radar to detect contested planets, value-based target selection, and sun avoidance
def make_guide_baseline_agent(
    early_off=True,
    early_off_until=50,
    angle_threshold=0.1,
    use_capture_filter=False,
    enemy_margin=5,
    use_estimate_defense=True,
    estimate_scale=0.8,
    use_enemy_radar=True,
    radar_angle_threshold=0.15,
    radar_start_step=0,
    wait_margin=0,
    use_best_target=True,
    target_score_type="prod_dist",
    target_phase_step=0,
    sun_buffer=0.5,
):
    def baseline(obs):
        moves = []
        step = obs["step"]
        player = obs["player"]
        my_planets, targets = split_planets(obs)

        if not my_planets or not targets:
            return moves

        # Early game: all planets are fair game. After early_off_until: track reserved targets from own fleets
        if early_off and step < early_off_until:
            reserved_targets = set()
        else:
            reserved_targets = get_reserved_targets_stationary(
                obs,
                angle_threshold=angle_threshold,
                use_capture_filter=use_capture_filter,
            )

        # For each owned planet, try to launch an attack on an available target
        for source in my_planets:
            available_targets = [t for t in targets if t[0] not in reserved_targets]
            if not available_targets:
                continue

            # Optional: filter out targets that enemies appear to be attacking
            if use_enemy_radar and step >= radar_start_step:
                available_targets = filter_targets_by_enemy_radar(
                    obs,
                    available_targets,
                    angle_threshold=radar_angle_threshold,
                )
                if not available_targets:
                    continue

            # Choose target by best value or nearest, depending on phase
            if use_best_target and step >= target_phase_step:
                target = get_best_value_target(source, available_targets, score_type=target_score_type)
            else:
                target = get_nearest_target(source, available_targets)

            ships_needed = target[5] + 1

            # If target is enemy-owned, add margin for enemy defense and optionally estimate defense strength
            if target[1] not in (-1, player):
                ships_needed += enemy_margin
                if use_estimate_defense:
                    estimated = estimate_target_defense(source, target, ships_needed)
                    ships_needed = max(ships_needed, math.ceil(estimated * estimate_scale) + 1)

            # Skip if we don't have enough ships (respecting wait_margin for safety)
            if wait_margin > 0 and ships_needed > source[5] - wait_margin:
                continue

            # Launch only if we have sufficient ships and path doesn't hit sun
            if source[5] >= ships_needed:
                angle = get_angle(source, target)
                if not path_hits_sun(source, angle, buffer=sun_buffer):
                    moves.append([source[0], angle, ships_needed])
                    reserved_targets.add(target[0])

        return moves

    baseline.__name__ = "guide_baseline_est08_radar015_prodDist_sun05"
    return baseline


# Instantiate the guide baseline agent with default parameters
guide_baseline_agent = make_guide_baseline_agent()


## My Agent v3: Guide Baseline With Lead Aiming

This agent is intentionally the guide baseline recipe with one navigation change: when it launches a fleet, it aims at the target planet's predicted intercept position instead of the target's current position.


In [5]:
# V3: exact guide baseline decision policy, but with lead-aimed launches.
# This isolates the value of planet-leading from the v2 config/scoring changes.
def make_v3_baseline_lead_agent(
    early_off=True,
    early_off_until=50,
    angle_threshold=0.1,
    use_capture_filter=False,
    enemy_margin=5,
    use_estimate_defense=True,
    estimate_scale=0.8,
    use_enemy_radar=True,
    radar_angle_threshold=0.15,
    radar_start_step=0,
    wait_margin=0,
    use_best_target=True,
    target_score_type="prod_dist",
    target_phase_step=0,
    sun_buffer=0.5,
):
    def baseline_lead(obs):
        moves = []
        step = obs["step"]
        player = obs["player"]
        my_planets, targets = split_planets(obs)

        if not my_planets or not targets:
            return moves

        if early_off and step < early_off_until:
            reserved_targets = set()
        else:
            reserved_targets = get_reserved_targets(
                obs,
                angle_threshold=angle_threshold,
                use_capture_filter=use_capture_filter,
            )

        for source in my_planets:
            available_targets = [t for t in targets if t[0] not in reserved_targets]
            if not available_targets:
                continue

            if use_enemy_radar and step >= radar_start_step:
                available_targets = filter_targets_by_enemy_radar(
                    obs,
                    available_targets,
                    angle_threshold=radar_angle_threshold,
                )
                if not available_targets:
                    continue

            if use_best_target and step >= target_phase_step:
                target = get_best_value_target(source, available_targets, score_type=target_score_type)
            else:
                target = get_nearest_target(source, available_targets)

            ships_needed = target[5] + 1

            if target[1] not in (-1, player):
                ships_needed += enemy_margin
                if use_estimate_defense:
                    estimated = estimate_target_defense(source, target, ships_needed)
                    ships_needed = max(ships_needed, math.ceil(estimated * estimate_scale) + 1)

            if wait_margin > 0 and ships_needed > source[5] - wait_margin:
                continue

            if source[5] >= ships_needed:
                angle = get_lead_angle(source, target, ships_needed, obs)
                if not path_hits_sun(source, angle, buffer=sun_buffer):
                    moves.append([source[0], angle, ships_needed])
                    reserved_targets.add(target[0])

        return moves

    baseline_lead.__name__ = "v3_baseline_lead_est08_radar015_prodDist_sun05"
    return baseline_lead


v3_baseline_lead_agent = make_v3_baseline_lead_agent()


## My Agent v1

The main changes are intentionally small and testable:

- `roi_target_score`: favors high production, low ship cost, closer distance, and neutral planets early.
- `source_reserve`: keeps ships at home, especially on productive planets.
- `defense_first`: uses spare ships to reinforce owned planets that enemy fleets seem to be targeting.
- `max_send_fraction`: prevents a single source planet from dumping its whole stockpile into one attack.

In [5]:
# Configuration dictionary for my_agent v1
# Tuned parameters balancing offense, defense, and resource management
MY_AGENT_CONFIG = {
    "early_off_until": 50,              # Step threshold: before this, no target reservation
    "reserved_angle": 0.10,             # Angle threshold for inferring fleet targets
    "radar_angle": 0.14,                # Angle threshold for enemy fleet detection
    "sun_buffer": 0.5,                 # Safety margin when checking sun collision
    "enemy_margin": 5,                  # Extra ships for enemy-owned planet attacks
    "estimate_scale": 0.8,              # Scale factor for estimated defense calculations
    "source_base_reserve": 0,           # Minimum ships to keep at home per planet
    "source_prod_reserve": 0.25,         # Multiplier for production-based reserves
    "max_send_fraction": 0.9,          # Max fraction of planet's ships to send in one attack
    "defense_first": True,              # Enable defensive reinforcement
    "defense_min_step": 60,             # Step threshold before defense kicks in
    "defense_extra": 1,                 # Extra ships to add when defending

    # ROI target scoring knobs
    "neutral_bonus_until": 70,          # Step threshold for neutral planet bonus
    "neutral_target_bonus": 1.25,       # Multiplier for neutral planets during early expansion
    "enemy_target_bonus": 1.0,         # Multiplier for enemy-owned planets
    "production_offset": 0.75,          # Added to target production before scoring
    "enemy_pressure_weight": 0.35,      # Penalty per incoming enemy ship toward the target
    "distance_penalty_base": 1.0,       # Base term in distance penalty
    "distance_penalty_scale": 24.0,     # Larger values make far targets less harshly penalized
    "contested_neutral_score": -1e9,    # Score for neutral planets enemies will likely capture first
    "min_attack_score": -1e8,           # Offensive target scores at/below this are skipped

}


# Calculate minimum ships to keep reserved at a source planet
# Scales with production: high-production planets reserve more
def source_reserve(source, config):
    return int(math.ceil(config["source_base_reserve"] + source[6] * config["source_prod_reserve"]))


# Calculate how many ships can be sent from a source without violating reserves and fraction limits
# Caps outflow at both reserve threshold and max_send_fraction to prevent over-commitment
def ships_available_to_send(source, config):
    reserve = source_reserve(source, config)
    fraction_cap = int(math.floor(source[5] * config["max_send_fraction"]))
    return max(0, min(source[5] - reserve, fraction_cap))


# Compute ROI score for a target: balances production value, capture cost, distance, and enemy pressure
# Returns very negative score (-1e9) if planet is neutral and under heavy enemy fire (will be lost anyway)
# Applies bonuses for neutral planets early-game and enemy-owned planets for conquest priority
def roi_target_score(source, target, obs, enemy_incoming, config):
    player = obs["player"]
    step = obs["step"]
    dist = max(distance(source, target), 1e-6)
    owner = target[1]
    ships = target[5]
    prod = target[6]
    pressure = enemy_incoming.get(target[0], 0)

    if owner == -1 and pressure >= ships + 1:
        return config["contested_neutral_score"]

    if owner == -1:
        capture_cost = ships + 1
    else:
        estimated = estimate_target_defense(
            source, target, ships + 1 + config["enemy_margin"]
        )
        capture_cost = max(
            ships + 1 + config["enemy_margin"],
            math.ceil(estimated * config["estimate_scale"]) + 1,
        )

    neutral_bonus = config["neutral_target_bonus"] if owner == -1 and step < config["neutral_bonus_until"] else 1.0
    enemy_bonus = config["enemy_target_bonus"] if owner not in (-1, player) else 1.0

    value = (prod + config["production_offset"]) * neutral_bonus * enemy_bonus
    cost = capture_cost + config["enemy_pressure_weight"] * pressure
    dist_penalty = config["distance_penalty_base"] + dist / config["distance_penalty_scale"]

    return value / (cost * dist_penalty)
   
# Mimic baseline activity    
"""
def roi_target_score(source, target, obs, enemy_incoming, config):
    
    prod = target[6]
    dist = max(distance(source, target), 1e-6)
    return prod / (dist + 1e-6)
"""



# Select the target with the highest ROI score from available options
def choose_roi_target(source, targets, obs, enemy_incoming, config):
    return max(targets, key=lambda t: roi_target_score(source, t, obs, enemy_incoming, config))


# Add defensive moves to reinforce owned planets under enemy threat
# Finds threatened planets (incoming enemy ships > current defense), sorts by urgency (largest deficit first)
# Sends reinforcements from nearest friendly planets, respecting reserves and sun safety
def add_defense_moves(obs, my_planets, moves, launched_by_source, config):
    if not config["defense_first"] or obs["step"] < config["defense_min_step"]:
        return

    player = obs["player"]
    enemy_incoming = incoming_ships_by_planet(obs, enemy_only=True, angle_threshold=config["radar_angle"])
    planet_by_id = {p[0]: p for p in my_planets}

    threatened = []
    for planet in my_planets:
        incoming = enemy_incoming.get(planet[0], 0)
        if incoming <= 0:
            continue

        deficit = incoming + config["defense_extra"] - planet[5]
        if deficit > 0:
            threatened.append((deficit, planet))

    threatened.sort(reverse=True, key=lambda item: item[0])

    for deficit, target in threatened:
        donors = sorted(
            [p for p in my_planets if p[0] != target[0]],
            key=lambda p: distance(p, target),
        )

        for source in donors:
            already_launched = launched_by_source.get(source[0], 0)
            effective_source = list(source)
            effective_source[5] = max(0, source[5] - already_launched)
            sendable = ships_available_to_send(effective_source, config)
            ships = min(deficit, sendable)

            if ships <= 0:
                continue

            angle = get_lead_angle(source, target, ships, obs)
            if path_hits_sun(source, angle, buffer=config["sun_buffer"]):
                continue

            moves.append([source[0], angle, ships])
            launched_by_source[source[0]] = launched_by_source.get(source[0], 0) + ships
            deficit -= ships

            if deficit <= 0:
                break


# Factory function to create a configurable agent instance
# The returned agent function uses MY_AGENT_CONFIG or custom config to decide all moves
def make_my_agent(config=None):
    merged_config = dict(MY_AGENT_CONFIG)
    if config is not None:
        merged_config.update(config)
    config = merged_config

    # Main agent function: evaluates current observation and returns list of [source_id, angle, ships]
    # Executes in order: defense first (if enabled), then offensive launches using ROI targeting
    def my_agent(obs):
        moves = []
        player = obs["player"]
        step = obs["step"]
        my_planets, targets = split_planets(obs)

        if not my_planets or not targets:
            return moves

        # Track which planets have already launched fleets this turn
        launched_by_source = {}
        add_defense_moves(obs, my_planets, moves, launched_by_source, config)

        # Early game: ignore in-flight reservations. After threshold: avoid duplicate targeting
        if step < config["early_off_until"]:
            reserved_targets = set()
        else:
            reserved_targets = get_reserved_targets(obs, angle_threshold=config["reserved_angle"], use_capture_filter=False)

        # Analyze incoming enemy fleets to detect contested planets
        enemy_incoming = incoming_ships_by_planet(obs, enemy_only=True, angle_threshold=config["radar_angle"])

        # Process planets in order of production + stockpile (high-value planets first)
        for source in sorted(my_planets, key=lambda p: (p[6], p[5]), reverse=True):
            already_launched = launched_by_source.get(source[0], 0)
            effective_source = list(source)
            effective_source[5] = max(0, source[5] - already_launched)
            sendable = ships_available_to_send(effective_source, config)

            if sendable <= 0:
                continue

            available_targets = [t for t in targets if t[0] not in reserved_targets]
            if not available_targets:
                continue

            # Choose best ROI target; skip if score is too low (contested by enemies)
            target = choose_roi_target(source, available_targets, obs, enemy_incoming, config)
            if roi_target_score(source, target, obs, enemy_incoming, config) <= config["min_attack_score"]:
                continue

            # Determine ships needed: more for enemy planets, less for neutral
            if target[1] == -1:
                ships_needed = target[5] + 1
            else:
                estimated = estimate_target_defense(source, target, target[5] + 1 + config["enemy_margin"])
                ships_needed = max(
                    target[5] + 1 + config["enemy_margin"],
                    math.ceil(estimated * config["estimate_scale"]) + 1,
                )

            # Only launch if we have enough ships after reserves
            if ships_needed > sendable:
                continue

            # Final safety check: avoid sun
            angle = get_lead_angle(source, target, ships_needed, obs)
            if path_hits_sun(source, angle, buffer=config["sun_buffer"]):
                continue

            moves.append([source[0], angle, ships_needed])
            launched_by_source[source[0]] = launched_by_source.get(source[0], 0) + ships_needed
            reserved_targets.add(target[0])

        return moves

    my_agent.__name__ = "my_roi_reserve_defense_agent_v1"
    return my_agent


# Instantiate my_agent with default configuration
agent = make_my_agent()


In [6]:
# Quick check: these are the ROI knobs you can tune without editing roi_target_score().
ROI_CONFIG_KEYS = [
    "neutral_bonus_until",
    "neutral_target_bonus",
    "enemy_target_bonus",
    "production_offset",
    "enemy_pressure_weight",
    "distance_penalty_base",
    "distance_penalty_scale",
    "contested_neutral_score",
    "min_attack_score",
]
{key: MY_AGENT_CONFIG[key] for key in ROI_CONFIG_KEYS}


{'neutral_bonus_until': 70,
 'neutral_target_bonus': 1.25,
 'enemy_target_bonus': 1.0,
 'production_offset': 0.75,
 'enemy_pressure_weight': 0.35,
 'distance_penalty_base': 1.0,
 'distance_penalty_scale': 24.0,
 'contested_neutral_score': -1000000000.0,
 'min_attack_score': -100000000.0}

## My Agent v2 - Phase Allocator

This version moves away from a noisy all-in-one ROI formula. Instead, it uses a global move allocator:

- Build many possible moves from every owned planet to plausible targets.
- Score candidates with a simple baseline-like signal first: production over distance.
- Add only small, configurable nudges for cost, phase, enemy pressure, and ownership.
- Pick the best non-conflicting moves across the whole board instead of letting each planet greedily choose alone.

The main idea is to keep the baseline's fast expansion behavior while improving coordination and avoiding obviously bad launches.


In [7]:
# My Agent v2: global phase allocator
# Different from v1: simple baseline-like target scoring, then global move selection.
V2_CONFIG = {
    # Phase controls
    "opening_until": 90,               # Prefer neutral expansion before this step
    "enemy_attack_after": 90,          # Do not attack enemy planets too early unless no neutrals exist
    "reservation_after": 45,           # Start avoiding targets already pursued by our fleets

    # Fleet inference / safety
    "reserved_angle": 0.10,
    "radar_angle": 0.14,
    "sun_buffer": 0.5,

    # Source reserves by phase. Keep these low; tempo matters a lot.
    "opening_reserve": 0,
    "mid_reserve": 1,
    "late_reserve": 2,
    "production_reserve_scale": 0.10,
    "opening_max_send_fraction": 1.0,
    "mid_max_send_fraction": 0.90,
    "late_max_send_fraction": 0.82,

    # Candidate generation
    "max_targets_per_source": 8,       # Keep candidate set focused and stable
    "max_moves_per_turn": 999,         # Set lower, e.g. 3-6, if launches look too scattered
    "allow_duplicate_targets": False,  # False imitates the guide's reservation behavior

    # Scoring modes to try: "baseline", "cost_aware", "payback"
    "score_mode": "baseline",          # 4p baseline selected from config comparison
    "neutral_bonus": 1.35,
    "enemy_bonus_mid": 0.65,
    "enemy_bonus_late": 1.10,
    "cost_weight": 0.25,               # Used by cost_aware/payback modes
    "pressure_weight": 0.30,
    "distance_scale": 24.0,
    "production_offset": 0.25,

    # Ships needed
    "neutral_overkill": 1,
    "enemy_margin": 5,
    "estimate_scale": 0.8,

    # Optional defense. Conservative by default, because too much defense kills expansion.
    "defend_after": 120,
    "defense_extra": 1,
    "max_defense_moves": 2,
}


def v2_phase_reserve(source, step, config):
    if step < config["opening_until"]:
        base = config["opening_reserve"]
    elif step < 140:
        base = config["mid_reserve"]
    else:
        base = config["late_reserve"]

    return int(math.ceil(base + source[6] * config["production_reserve_scale"]))


def v2_max_send_fraction(step, config):
    if step < config["opening_until"]:
        return config["opening_max_send_fraction"]
    if step < 140:
        return config["mid_max_send_fraction"]
    return config["late_max_send_fraction"]


def v2_sendable_ships(source, step, already_launched, config):
    remaining = max(0, source[5] - already_launched)
    reserve = v2_phase_reserve(source, step, config)
    fraction_cap = int(math.floor(remaining * v2_max_send_fraction(step, config)))
    return max(0, min(remaining - reserve, fraction_cap))


def v2_ships_needed(source, target, player, config):
    if target[1] == -1:
        return target[5] + config["neutral_overkill"]

    base = target[5] + 1 + config["enemy_margin"]
    estimated = estimate_target_defense(source, target, base)
    return max(base, math.ceil(estimated * config["estimate_scale"]) + 1)


def v2_target_phase_allowed(target, step, any_neutrals, player, config):
    # The baseline wins by taking neutral production quickly. Do not get distracted too early.
    if target[1] == -1:
        return True
    if not any_neutrals:
        return True
    return step >= config["enemy_attack_after"]


def v2_candidate_score(source, target, obs, ships_needed, enemy_incoming, config):
    player = obs["player"]
    step = obs["step"]
    dist = max(distance(source, target), 1e-6)
    prod = target[6]
    owner = target[1]
    pressure = enemy_incoming.get(target[0], 0)

    # Avoid obvious neutral races where enemy fleets are already enough to capture first.
    # This is deliberately softer than v1: it only blocks the most clearly wasted moves.
    if owner == -1 and pressure >= target[5] + config["neutral_overkill"] + 3:
        return -1e12

    if owner == -1:
        ownership_bonus = config["neutral_bonus"] if step < config["opening_until"] else 1.0
    else:
        ownership_bonus = config["enemy_bonus_late"] if step >= 140 else config["enemy_bonus_mid"]

    baseline_score = (prod + config["production_offset"]) * ownership_bonus / dist

    if config["score_mode"] == "baseline":
        return baseline_score

    pressure_cost = config["pressure_weight"] * pressure
    cost_term = max(1.0, ships_needed + pressure_cost)

    if config["score_mode"] == "cost_aware":
        # Mildly penalize expensive targets without letting cost dominate everything.
        return baseline_score / (cost_term ** config["cost_weight"])

    if config["score_mode"] == "payback":
        # Roughly: how quickly this planet pays back the capture cost, with distance drag.
        payback = (prod + config["production_offset"]) / cost_term
        distance_penalty = 1.0 + dist / config["distance_scale"]
        return ownership_bonus * payback / distance_penalty

    raise ValueError(f"Unknown v2 score_mode: {config['score_mode']}")


def v2_build_candidates(obs, my_planets, targets, reserved_targets, launched_by_source, config):
    player = obs["player"]
    step = obs["step"]
    any_neutrals = any(t[1] == -1 for t in targets)
    enemy_incoming = incoming_ships_by_planet(obs, enemy_only=True, angle_threshold=config["radar_angle"])
    candidates = []

    for source in my_planets:
        already = launched_by_source.get(source[0], 0)
        sendable = v2_sendable_ships(source, step, already, config)
        if sendable <= 0:
            continue

        plausible_targets = []
        for target in targets:
            if not config["allow_duplicate_targets"] and target[0] in reserved_targets:
                continue
            if not v2_target_phase_allowed(target, step, any_neutrals, player, config):
                continue

            ships_needed = v2_ships_needed(source, target, player, config)
            if ships_needed > sendable:
                continue

            angle = get_lead_angle(source, target, ships_needed, obs)
            if path_hits_sun(source, angle, buffer=config["sun_buffer"]):
                continue

            score = v2_candidate_score(source, target, obs, ships_needed, enemy_incoming, config)
            if score <= -1e11:
                continue

            plausible_targets.append((score, source, target, angle, ships_needed))

        plausible_targets.sort(reverse=True, key=lambda item: item[0])
        candidates.extend(plausible_targets[:config["max_targets_per_source"]])

    candidates.sort(reverse=True, key=lambda item: item[0])
    return candidates


def v2_add_late_defense(obs, my_planets, moves, launched_by_source, config):
    if obs["step"] < config["defend_after"]:
        return

    enemy_incoming = incoming_ships_by_planet(obs, enemy_only=True, angle_threshold=config["radar_angle"])
    threatened = []

    for planet in my_planets:
        incoming = enemy_incoming.get(planet[0], 0)
        deficit = incoming + config["defense_extra"] - planet[5]
        if deficit > 0:
            threatened.append((deficit, planet))

    threatened.sort(reverse=True, key=lambda item: item[0])
    defense_moves = 0

    for deficit, target in threatened:
        donors = sorted(
            [p for p in my_planets if p[0] != target[0]],
            key=lambda p: distance(p, target),
        )

        for source in donors:
            already = launched_by_source.get(source[0], 0)
            sendable = v2_sendable_ships(source, obs["step"], already, config)
            ships = min(deficit, sendable)
            if ships <= 0:
                continue

            angle = get_lead_angle(source, target, ships, obs)
            if path_hits_sun(source, angle, buffer=config["sun_buffer"]):
                continue

            moves.append([source[0], angle, ships])
            launched_by_source[source[0]] = launched_by_source.get(source[0], 0) + ships
            defense_moves += 1
            break

        if defense_moves >= config["max_defense_moves"]:
            break


def make_my_agent_v2(config=None):
    merged_config = dict(V2_CONFIG)
    if config is not None:
        merged_config.update(config)
    config = merged_config

    def my_agent_v2(obs):
        moves = []
        step = obs["step"]
        my_planets, targets = split_planets(obs)

        if not my_planets or not targets:
            return moves

        launched_by_source = {}

        if step >= config["reservation_after"]:
            reserved_targets = get_reserved_targets(
                obs,
                angle_threshold=config["reserved_angle"],
                use_capture_filter=False,
            )
        else:
            reserved_targets = set()

        candidates = v2_build_candidates(
            obs=obs,
            my_planets=my_planets,
            targets=targets,
            reserved_targets=reserved_targets,
            launched_by_source=launched_by_source,
            config=config,
        )

        used_sources = set()
        used_targets = set(reserved_targets)

        for score, source, target, angle, ships_needed in candidates:
            if len(moves) >= config["max_moves_per_turn"]:
                break
            if source[0] in used_sources:
                continue
            if not config["allow_duplicate_targets"] and target[0] in used_targets:
                continue

            already = launched_by_source.get(source[0], 0)
            if ships_needed > v2_sendable_ships(source, step, already, config):
                continue

            moves.append([source[0], angle, ships_needed])
            used_sources.add(source[0])
            used_targets.add(target[0])
            launched_by_source[source[0]] = already + ships_needed

        # Defense happens after offense by default, using leftovers only.
        # This protects late weak planets without sacrificing the opening snowball.
        v2_add_late_defense(obs, my_planets, moves, launched_by_source, config)

        return moves

    my_agent_v2.__name__ = f"my_phase_allocator_v2_{config['score_mode']}"
    return my_agent_v2


# Make v2 the default agent for the evaluation cells below.
agent_v2 = make_my_agent_v2()
agent = agent_v2


## Quick Smoke Test

This does not prove strength; it just checks that `agent(obs)` returns legal-looking moves on a tiny fake observation.

In [8]:
# Test observation with a few planets
# Planet format: [id, owner, x, y, ?, ships, production]
# Owner: 0 = me, -1 = neutral, 1 = enemy
fake_obs = {
    "step": 30,
    "player": 0,
    "planets": [
        [0, 0, 20.0, 20.0, 0, 45, 3],      # My planet at (20,20): 45 ships, 3 prod
        [1, -1, 32.0, 22.0, 0, 12, 2],     # Neutral planet: 12 ships, 2 prod
        [2, 1, 70.0, 70.0, 0, 28, 4],      # Enemy planet: 28 ships, 4 prod
        [3, 0, 24.0, 62.0, 0, 18, 1],      # My planet at (24,62): 18 ships, 1 prod
    ],
    "fleets": [],
}

# Execute agent on test observation and display moves
moves = agent(fake_obs)
moves


[[0, 0.16514867741462683, 13]]

## Evaluation Harness

Run these cells locally or on Kaggle after installing/using `kaggle_environments`. Use more seeds once the smoke test looks healthy.

In [7]:
# Count total ships (both on planets and in fleets) owned by a player
# Used to determine final strength and winner
def total_ships(obs, player_id):
    planet_ships = sum(p[5] for p in obs["planets"] if p[1] == player_id)
    fleet_ships = sum(f[6] for f in obs["fleets"] if f[1] == player_id)
    return planet_ships + fleet_ships


# Run a tournament between agents and collect statistics
# Randomizes starting positions (my_agent may start as any player)
# Returns a list of rows (or DataFrame if pandas available) with seed, position, winner, rank, ship totals, etc.
def evaluate_agents(agents, my_agent, seeds=range(20)):
    if make is None:
        raise RuntimeError("kaggle_environments is not available in this kernel")

    rows = []
    n_players = len(agents)

    for seed in seeds:
        # Shuffle agent lineup so my_agent's position varies by seed
        lineup = list(agents)
        random.Random(seed).shuffle(lineup)
        my_position = lineup.index(my_agent)

        # Create environment and run game
        env = make("orbit_wars", debug=False, configuration={"seed": seed})
        env.run(lineup)

        # Extract final state: total ships per player
        final_states = env.steps[-1]
        totals = [total_ships(final_states[player_id].observation, player_id) for player_id in range(n_players)]
        max_total = max(totals)
        winners = [pid for pid, ships in enumerate(totals) if ships == max_total]
        winner = winners[0] if len(winners) == 1 else -1  # -1 if tie

        my_total = totals[my_position]
        best_other = max(ships for pid, ships in enumerate(totals) if pid != my_position)
        rank = sorted(totals, reverse=True).index(my_total) + 1

        # Record game outcome
        rows.append({
            "seed": seed,
            "my_position": my_position,
            "winner": winner,
            "my_won": winner == my_position,
            "my_rank": rank,
            "my_total_ships": my_total,
            "best_other_ships": best_other,
            "ship_margin_vs_best_other": my_total - best_other,
            "final_turns": len(env.steps),
        })

    if pd is not None:
        return pd.DataFrame(rows)
    return rows


# Summarize tournament results: compute win rate, average rank, avg ships, margin vs best opponent, avg turns
# Returns dict (if no pandas) or Series with aggregated stats
def summarize(df):
    if pd is None:
        wins = sum(row["my_won"] for row in df)
        return {"games": len(df), "wins": wins, "win_rate": wins / len(df)}

    return pd.Series({
        "games": len(df),
        "win_rate": df["my_won"].mean(),
        "avg_rank": df["my_rank"].mean(),
        "avg_ships": df["my_total_ships"].mean(),
        "avg_margin": df["ship_margin_vs_best_other"].mean(),
        "avg_turns": df["final_turns"].mean(),
    })


## V2 Fine Tuning

Use these cells to tune v2 without editing the agent body. Start with the presets, then sweep one knob at a time using the same seed range so the comparisons are less noisy.

Good first knobs to test: `score_mode`, `opening_until`, `enemy_attack_after`, reserves, `neutral_bonus`, and `cost_weight`.


In [ ]:
# V2 tuning helpers
# These functions run controlled comparisons against the extracted guide baseline.
# Keep seed ranges small while exploring, then rerun promising configs with 100+ seeds.

V2_PRESETS = {
    "tempo_baseline": {},
    "pure_baseline_cloneish": {
        "score_mode": "baseline",
        "opening_until": 50,
        "enemy_attack_after": 50,
        "reservation_after": 50,
        "neutral_bonus": 1.0,
        "enemy_bonus_mid": 1.0,
        "enemy_bonus_late": 1.0,
        "production_offset": 0.0,
        "opening_reserve": 0,
        "mid_reserve": 0,
        "late_reserve": 1,
        "production_reserve_scale": 0.0,
        "opening_max_send_fraction": 1.0,
        "mid_max_send_fraction": 1.0,
        "late_max_send_fraction": 0.95,
        "defend_after": 9999,
    },
    "neutral_snowball": {
        "score_mode": "baseline",
        "opening_until": 90,
        "enemy_attack_after": 90,
        "neutral_bonus": 1.35,
        "enemy_bonus_mid": 0.65,
        "opening_reserve": 0,
        "production_reserve_scale": 0.10,
        "opening_max_send_fraction": 1.0,
        "defend_after": 120,
    },
    "light_cost_awareness": {
        "score_mode": "cost_aware",
        "cost_weight": 0.12,
        "neutral_bonus": 1.15,
        "enemy_bonus_mid": 0.80,
        "production_offset": 0.10,
        "opening_reserve": 0,
        "production_reserve_scale": 0.10,
    },
    "payback_cautious": {
        "score_mode": "payback",
        "opening_until": 80,
        "enemy_attack_after": 95,
        "cost_weight": 0.20,
        "neutral_bonus": 1.25,
        "enemy_bonus_mid": 0.60,
        "distance_scale": 30.0,
        "defend_after": 120,
    },
    "late_enemy_cleaner": {
        "score_mode": "baseline",
        "opening_until": 70,
        "enemy_attack_after": 110,
        "enemy_bonus_mid": 0.50,
        "enemy_bonus_late": 1.35,
        "enemy_margin": 4,
        "estimate_scale": 0.75,
        "defend_after": 100,
    },
    "no_defense_fast": {
        "score_mode": "baseline",
        "opening_until": 80,
        "enemy_attack_after": 80,
        "opening_reserve": 0,
        "mid_reserve": 0,
        "production_reserve_scale": 0.0,
        "opening_max_send_fraction": 1.0,
        "mid_max_send_fraction": 0.98,
        "defend_after": 9999,
    },
}


def make_named_v2_agent(name, overrides=None):
    cfg = dict(V2_CONFIG)
    if overrides:
        cfg.update(overrides)
    tuned_agent = make_my_agent_v2(cfg)
    tuned_agent.__name__ = f"v2_{name}"
    return tuned_agent, cfg


def summarize_eval_results(results):
    summary = summarize(results)
    if pd is not None and hasattr(summary, "to_dict"):
        return summary.to_dict()
    return dict(summary)


def evaluate_v2_config(name, overrides=None, seeds=range(20), mode="2p", opponent="baseline"):
    candidate, cfg = make_named_v2_agent(name, overrides)

    if opponent == "baseline":
        if mode == "2p":
            lineup = [candidate, guide_baseline_agent]
        elif mode == "4p":
            lineup = [candidate, guide_baseline_agent, guide_baseline_agent, guide_baseline_agent]
        else:
            raise ValueError("mode must be '2p' or '4p'")
    elif opponent == "random":
        if mode == "2p":
            lineup = [candidate, "random"]
        elif mode == "4p":
            lineup = [candidate, "random", "random", "random"]
        else:
            raise ValueError("mode must be '2p' or '4p'")
    else:
        raise ValueError("opponent must be 'baseline' or 'random'")

    results = evaluate_agents(lineup, my_agent=candidate, seeds=seeds)
    row = {
        "name": name,
        "mode": mode,
        "opponent": opponent,
        **summarize_eval_results(results),
    }

    # Include the highest-signal config values in the results table.
    for key in [
        "score_mode",
        "opening_until",
        "enemy_attack_after",
        "reservation_after",
        "neutral_bonus",
        "enemy_bonus_mid",
        "enemy_bonus_late",
        "cost_weight",
        "opening_reserve",
        "mid_reserve",
        "production_reserve_scale",
        "opening_max_send_fraction",
        "defend_after",
    ]:
        row[key] = cfg[key]

    return row, results, cfg


def evaluate_v2_presets(presets=None, seeds=range(20), mode="2p", opponent="baseline"):
    presets = V2_PRESETS if presets is None else presets
    rows = []

    iterator = presets.items()
    if tqdm is not None:
        iterator = tqdm(list(iterator), desc=f"V2 presets vs {opponent} {mode}")

    for name, overrides in iterator:
        row, _, _ = evaluate_v2_config(
            name=name,
            overrides=overrides,
            seeds=seeds,
            mode=mode,
            opponent=opponent,
        )
        rows.append(row)

    if pd is not None:
        return pd.DataFrame(rows).sort_values(["win_rate", "avg_margin"], ascending=False)
    return sorted(rows, key=lambda r: (r["win_rate"], r.get("avg_margin", 0)), reverse=True)


def sweep_v2_param(param, values, base_overrides=None, seeds=range(20), mode="2p", opponent="baseline"):
    rows = []
    base_overrides = {} if base_overrides is None else dict(base_overrides)

    iterator = values
    if tqdm is not None:
        iterator = tqdm(list(values), desc=f"Sweeping {param}")

    for value in iterator:
        overrides = dict(base_overrides)
        overrides[param] = value
        row, _, _ = evaluate_v2_config(
            name=f"{param}_{value}",
            overrides=overrides,
            seeds=seeds,
            mode=mode,
            opponent=opponent,
        )
        row[param] = value
        rows.append(row)

    if pd is not None:
        return pd.DataFrame(rows).sort_values(["win_rate", "avg_margin"], ascending=False)
    return sorted(rows, key=lambda r: (r["win_rate"], r.get("avg_margin", 0)), reverse=True)


def small_v2_grid(seeds=range(20), mode="2p", opponent="baseline"):
    # Compact grid over the knobs most likely to explain wins/losses.
    # This is intentionally small enough to run interactively.
    rows = []
    score_modes = ["baseline", "cost_aware"]
    opening_untils = [50, 70, 90]
    enemy_attack_afters = [55, 80, 110]
    reserve_scales = [0.0, 0.1, 0.2]

    combos = [
        (score_mode, opening_until, enemy_attack_after, reserve_scale)
        for score_mode in score_modes
        for opening_until in opening_untils
        for enemy_attack_after in enemy_attack_afters
        for reserve_scale in reserve_scales
        if enemy_attack_after >= opening_until - 20
    ]

    iterator = combos
    if tqdm is not None:
        iterator = tqdm(combos, desc="Small V2 grid")

    for score_mode, opening_until, enemy_attack_after, reserve_scale in iterator:
        overrides = {
            "score_mode": score_mode,
            "opening_until": opening_until,
            "enemy_attack_after": enemy_attack_after,
            "production_reserve_scale": reserve_scale,
            "opening_reserve": 0,
            "mid_reserve": 0 if reserve_scale == 0.0 else 1,
            "defend_after": 9999,
        }
        name = f"{score_mode}_open{opening_until}_enemy{enemy_attack_after}_res{reserve_scale}"
        row, _, _ = evaluate_v2_config(name, overrides, seeds=seeds, mode=mode, opponent=opponent)
        rows.append(row)

    if pd is not None:
        return pd.DataFrame(rows).sort_values(["win_rate", "avg_margin"], ascending=False)
    return sorted(rows, key=lambda r: (r["win_rate"], r.get("avg_margin", 0)), reverse=True)


### Suggested Tuning Order

1. Run presets against the baseline on 20 seeds.
2. Take the best preset and sweep one parameter at a time.
3. Confirm the top 2-3 configs with 100+ seeds and both 2-player and 4-player matches.


In [ ]:
# Suggested first runs. Uncomment one block at a time.

# 1) Quick preset comparison against the guide baseline.
preset_results_2p = evaluate_v2_presets(seeds=range(20), mode="2p", opponent="baseline")
preset_results_2p


V2 presets vs baseline 2p: 100%|██████████| 7/7 [03:31<00:00, 30.24s/it]


,name,mode,opponent,games,win_rate,avg_rank,avg_ships,avg_margin,avg_turns,score_mode,...,reservation_after,neutral_bonus,enemy_bonus_mid,enemy_bonus_late,cost_weight,opening_reserve,mid_reserve,production_reserve_scale,opening_max_send_fraction,defend_after
1,pure_baseline_cloneish,2p,baseline,20.0,0.35,1.65,1432.25,-4046.85,263.10,baseline,...,50,1.00,1.00,1.00,0.25,0,0,0.0,1.00,9999
3,light_cost_awareness,2p,baseline,20.0,0.30,1.70,2111.75,-2755.80,247.15,cost_aware,...,45,1.15,0.80,1.10,0.12,0,1,0.1,0.95,85
6,no_defense_fast,2p,baseline,20.0,0.20,1.80,771.20,-5010.25,240.45,baseline,...,45,1.20,0.85,1.10,0.25,0,0,0.0,1.00,9999
4,payback_cautious,2p,baseline,20.0,0.20,1.80,884.35,-5993.10,245.50,payback,...,45,1.25,0.60,1.10,0.20,0,1,0.2,0.95,120
2,neutral_snowball,2p,baseline,20.0,0.20,1.80,1744.30,-6520.70,284.00,baseline,...,45,1.35,0.65,1.10,0.25,0,1,0.1,1.00,120
5,late_enemy_cleaner,2p,baseline,20.0,0.15,1.85,1564.75,-4847.15,237.35,baseline,...,45,1.20,0.50,1.35,0.25,0,1,0.2,0.95,100
0,tempo_baseline,2p,baseline,20.0,0.15,1.85,2029.40,-6451.10,278.15,baseline,...,45,1.20,0.85,1.10,0.25,0,1,0.2,0.95,85


In [52]:
# 2) If a preset looks good, sweep a single knob around it.
best_base = V2_PRESETS["pure_baseline_cloneish"]
sweep_v2_param("enemy_attack_after", [50, 65, 80, 95, 110, 130], base_overrides=best_base, seeds=range(30))

Sweeping enemy_attack_after: 100%|██████████| 6/6 [03:41<00:00, 36.90s/it]


,name,mode,opponent,games,win_rate,avg_rank,avg_ships,avg_margin,avg_turns,score_mode,...,reservation_after,neutral_bonus,enemy_bonus_mid,enemy_bonus_late,cost_weight,opening_reserve,mid_reserve,production_reserve_scale,opening_max_send_fraction,defend_after
3,enemy_attack_after_95,2p,baseline,30.0,0.300000,1.700000,1045.566667,-6148.400000,255.700000,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999
5,enemy_attack_after_130,2p,baseline,30.0,0.266667,1.733333,1377.766667,-4750.166667,241.833333,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999
1,enemy_attack_after_65,2p,baseline,30.0,0.233333,1.766667,870.333333,-5312.433333,238.366667,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999
4,enemy_attack_after_110,2p,baseline,30.0,0.200000,1.800000,535.533333,-4739.066667,210.433333,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999
0,enemy_attack_after_50,2p,baseline,30.0,0.166667,1.833333,741.266667,-5630.566667,240.733333,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999
2,enemy_attack_after_80,2p,baseline,30.0,0.166667,1.833333,619.233333,-5931.066667,264.000000,baseline,...,50,1.0,1.0,1.0,0.25,0,0,0.0,1.0,9999


In [53]:
# 3) Run the compact grid when you want broader signal.
grid_results = small_v2_grid(seeds=range(20), mode="2p", opponent="baseline")
grid_results.head(10) if pd is not None else grid_results[:10]


Small V2 grid: 100%|██████████| 48/48 [19:48<00:00, 24.77s/it]


,name,mode,opponent,games,win_rate,avg_rank,avg_ships,avg_margin,avg_turns,score_mode,...,reservation_after,neutral_bonus,enemy_bonus_mid,enemy_bonus_late,cost_weight,opening_reserve,mid_reserve,production_reserve_scale,opening_max_send_fraction,defend_after
43,cost_aware_open90_enemy80_res0.1,2p,baseline,20.0,0.45,1.55,2300.85,-2017.80,249.60,cost_aware,...,45,1.2,0.85,1.1,0.25,0,1,0.1,0.95,9999
24,cost_aware_open50_enemy55_res0.0,2p,baseline,20.0,0.40,1.60,2150.95,-1179.60,215.40,cost_aware,...,45,1.2,0.85,1.1,0.25,0,0,0.0,0.95,9999
33,cost_aware_open70_enemy55_res0.0,2p,baseline,20.0,0.40,1.60,2498.20,-1545.20,265.00,cost_aware,...,45,1.2,0.85,1.1,0.25,0,0,0.0,0.95,9999
39,cost_aware_open70_enemy110_res0.0,2p,baseline,20.0,0.40,1.60,2907.25,-2088.80,276.65,cost_aware,...,45,1.2,0.85,1.1,0.25,0,0,0.0,0.95,9999
38,cost_aware_open70_enemy80_res0.2,2p,baseline,20.0,0.40,1.60,2642.80,-3361.00,260.90,cost_aware,...,45,1.2,0.85,1.1,0.25,0,1,0.2,0.95,9999
23,baseline_open90_enemy110_res0.2,2p,baseline,20.0,0.35,1.65,3744.80,-2676.35,272.10,baseline,...,45,1.2,0.85,1.1,0.25,0,1,0.2,0.95,9999
46,cost_aware_open90_enemy110_res0.1,2p,baseline,20.0,0.35,1.65,1657.50,-3026.55,234.35,cost_aware,...,45,1.2,0.85,1.1,0.25,0,1,0.1,0.95,9999
1,baseline_open50_enemy55_res0.1,2p,baseline,20.0,0.35,1.65,1168.90,-3130.50,215.90,baseline,...,45,1.2,0.85,1.1,0.25,0,1,0.1,0.95,9999
19,baseline_open90_enemy80_res0.1,2p,baseline,20.0,0.35,1.65,1374.15,-3847.35,257.75,baseline,...,45,1.2,0.85,1.1,0.25,0,1,0.1,0.95,9999
40,cost_aware_open70_enemy110_res0.1,2p,baseline,20.0,0.35,1.65,1257.70,-4358.80,240.45,cost_aware,...,45,1.2,0.85,1.1,0.25,0,1,0.1,0.95,9999


In [ ]:
# Suggested first runs. Uncomment one block at a time.

# 1) Quick preset comparison against the guide baseline.
# preset_results_4p = evaluate_v2_presets(seeds=range(20), mode="4p", opponent="baseline")
# preset_results_4p


V2 presets vs baseline 4p: 100%|██████████| 7/7 [06:17<00:00, 53.95s/it]


,name,mode,opponent,games,win_rate,avg_rank,avg_ships,avg_margin,avg_turns,score_mode,...,reservation_after,neutral_bonus,enemy_bonus_mid,enemy_bonus_late,cost_weight,opening_reserve,mid_reserve,production_reserve_scale,opening_max_send_fraction,defend_after
2,neutral_snowball,4p,baseline,20.0,0.15,2.00,3591.20,-3277.80,390.05,baseline,...,45,1.35,0.65,1.10,0.25,0,1,0.1,1.00,120
5,late_enemy_cleaner,4p,baseline,20.0,0.10,2.00,2384.20,-4277.40,355.85,baseline,...,45,1.20,0.50,1.35,0.25,0,1,0.2,0.95,100
1,pure_baseline_cloneish,4p,baseline,20.0,0.10,1.95,379.35,-5531.15,259.05,baseline,...,50,1.00,1.00,1.00,0.25,0,0,0.0,1.00,9999
4,payback_cautious,4p,baseline,20.0,0.10,1.95,791.35,-7544.10,311.95,payback,...,45,1.25,0.60,1.10,0.20,0,1,0.2,0.95,120
3,light_cost_awareness,4p,baseline,20.0,0.10,2.05,486.05,-7721.35,324.35,cost_aware,...,45,1.15,0.80,1.10,0.12,0,1,0.1,0.95,85
0,tempo_baseline,4p,baseline,20.0,0.05,2.00,218.35,-6679.15,303.00,baseline,...,45,1.20,0.85,1.10,0.25,0,1,0.2,0.95,85
6,no_defense_fast,4p,baseline,20.0,0.05,2.00,99.40,-6877.05,284.80,baseline,...,45,1.20,0.85,1.10,0.25,0,0,0.0,1.00,9999


In [ ]:

# 2) If a preset looks good, sweep a single knob around it.
best_base = V2_PRESETS["neutral_snowball"]
sweep_v2_param("enemy_attack_after", [50, 65, 80, 95, 110, 130], base_overrides=best_base, seeds=range(30))


In [ ]:

# 3) Run the compact grid when you want broader signal.
grid_results = small_v2_grid(seeds=range(20), mode="2p", opponent="baseline")
grid_results.head(10) if pd is not None else grid_results[:10]


In [ ]:
# 4p config selection after intercept aiming fix.
# Identical seeds, sorted by avg_margin first, then avg_ships, then avg_rank.
CONFIG_SELECTION_4P_RESULTS = [
    {
        "name": "neutral_snowball",
        "games": 30.0,
        "win_rate": 0.0667,
        "avg_rank": 2.00,
        "avg_ships": 832.20,
        "avg_margin": -6755.63,
        "avg_turns": 265.13,
    },
    {
        "name": "current_default_v2_pre_selection",
        "games": 30.0,
        "win_rate": 0.1000,
        "avg_rank": 2.03,
        "avg_ships": 474.67,
        "avg_margin": -8407.60,
        "avg_turns": 285.30,
    },
    {
        "name": "no_defense_fast",
        "games": 30.0,
        "win_rate": 0.0333,
        "avg_rank": 2.03,
        "avg_ships": 53.47,
        "avg_margin": -8701.83,
        "avg_turns": 278.30,
    },
    {
        "name": "exported_cost_aware_open90_enemy80_res0.1",
        "games": 30.0,
        "win_rate": 0.0333,
        "avg_rank": 2.13,
        "avg_ships": 63.07,
        "avg_margin": -9466.80,
        "avg_turns": 297.77,
    },
]

CONFIRMED_4P_BASELINE = {
    "name": "neutral_snowball_confirm100",
    "games": 100.0,
    "win_rate": 0.05,
    "avg_rank": 2.07,
    "avg_ships": 319.34,
    "avg_margin": -9296.75,
    "avg_turns": 297.52,
}

if pd is not None:
    display(pd.DataFrame(CONFIG_SELECTION_4P_RESULTS))
    display(pd.Series(CONFIRMED_4P_BASELINE))
else:
    CONFIG_SELECTION_4P_RESULTS, CONFIRMED_4P_BASELINE


In [8]:
# Start small while tuning, then increase to range(100+) for a serious read.
random_2p = evaluate_agents([agent, "random"], my_agent=agent, seeds=range(20))
baseline_2p = evaluate_agents([guide_baseline_agent, "random"], my_agent=guide_baseline_agent, seeds=range(20))
summarize(random_2p), summarize(baseline_2p)

NameError: name 'agent' is not defined

In [ ]:
# Isolated test for planet-leading effectiveness.
# Compare the original guide baseline against v3, which only changes launch aiming.
v3_2p = evaluate_agents([v3_baseline_lead_agent, guide_baseline_agent], my_agent=v3_baseline_lead_agent, seeds=range(20))
v3_4p = evaluate_agents([v3_baseline_lead_agent, guide_baseline_agent, guide_baseline_agent, guide_baseline_agent], my_agent=v3_baseline_lead_agent, seeds=range(20))
summarize(v3_2p), summarize(v3_4p)


In [11]:
# Head-to-head against the extracted guide baseline.
h2h_2p = evaluate_agents([agent, guide_baseline_agent], my_agent=agent, seeds=range(20))
h2h_4p = evaluate_agents([agent, guide_baseline_agent, guide_baseline_agent, guide_baseline_agent], my_agent=agent, seeds=range(20))
summarize(h2h_2p), summarize(h2h_4p)

(games           20.0
 win_rate         0.2
 avg_rank         1.8
 avg_ships      624.7
 avg_margin   -8451.8
 avg_turns      237.6
 dtype: float64,
 games           20.00
 win_rate         0.00
 avg_rank         2.15
 avg_ships        1.60
 avg_margin   -9573.95
 avg_turns      285.70
 dtype: float64)